# Лабораторная работа 5. 
Cистема, позволяющей проводить анализ имиджа бренда на основе текстовых данных, получаемых из социальных сетей

In [ ]:
!pip install --upgrade pip
!pip install numpy torch_directml datasets transformers

import numpy as np
import torch_directml
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

## Модель
Для задачи анализа эмоциональной окрашенности текста хорошо подойдет модель типа BERT. В данном случае используем ее оптимизированную версию RoBERTa (robust BERT), а именно реализацию [distilbert/distilroberta-base](https://huggingface.co/distilbert/distilroberta-base). Определим модель и девайс для обучения (используем DirectML, так как обучение шло на карте AMD).

In [ ]:
device = torch_directml.device()
print(f"Device: {device}")s

model_name = "distilbert/distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)

Для того чтобы модель понимала эмоциональную окраску текста, обучим ее на датасете [EleutherAI/twitter-sentiment](https://huggingface.co/datasets/EleutherAI/twitter-sentiment). Он содержит набор из 1,578,627 записей из Твиттера, размеченые на два класса: 
- 0 - Negative; 
- 1 - Positive.

In [ ]:
raw_dataset = load_dataset("EleutherAI/twitter-sentiment")

Сократим выборку, чтобы ускорить обучение

In [ ]:
total_samples = min(135000, len(raw_dataset["train"]))
full_train_sample = (
    raw_dataset["train"].shuffle(seed=42).select(range(total_samples))
)

split = full_train_sample.train_test_split(test_size=0.1, seed=42)

def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, max_length=128)

tokenized_ds = split.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Определим метрики

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

Проведем обучение на трех эпохах

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=False,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Сохраним веса

In [ ]:
model.save_pretrained("./fine_tuned_roberta")
tokenizer.save_pretrained("./fine_tuned_roberta")

## Первичное тестирование
Проведем первичное тестирование обученной модели на некторых моковых данных. Загрузим модель.

In [ ]:
model_path = "./fine_tuned_roberta"
device = torch_directml.device()

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)

Объявим классификатор

In [ ]:
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Определим моковые данные для теста

In [ ]:
labels_map = {"LABEL_0": "(-) Negative", "LABEL_1": "(+) Positive"}
test_phrases = [
    "I love how this brand cares about the environment!",
    "Just bought their new sneakers. Best purchase ever! 🔥",
    "The new update is absolutely terrible, everything is broken.",
    "Worst customer service experience in my life.",
    "The delivery was a bit late, but the product is high quality.",
    "I'm not sure if I like it, but it's definitely not bad.",
    "Disappointed with the packaging, it arrived damaged.",
    "Simply amazing service, highly recommend to everyone!",
    "The product arrived today. It's okay, I guess.",
]

Проведем тестирование

In [ ]:
print("\n" + "=" * 50)
print("TEST RESULTS:")
print("=" * 50)

for phrase in test_phrases:
    result = classifier(phrase)[0]
    label_id = result["label"]
    status = labels_map.get(label_id, label_id)
    confidence = result["score"]

    print(f"Sample: {phrase}")
    print(f"Result: {status} | Confidence: {confidence:.2%}")
    print("-" * 50)

Модель проявляет низкий уровень уверенности для нейтральных текстов (например, для последнего). Это связано с тем, что в обучающей выборке не было такого класса как нейтральный текст. Это полезно для анализа имиджа бренда, так как нивелирует влияние среднего мнения на заключение.

## Реальное тестирование
Теперь протестируем модель на реальных данных. Проведем анализ имиджа бренда Apple. Для этого необходимо взять корпус тестовых данных, относящихся к бренду, из социальных сетей. Для этого хорошо подойдет датасет [apple_twitter_sentiment_texts](https://www.kaggle.com/datasets/seriousran/appletwittersentimenttexts), он содержит размеченные твиты про компанию Apple. То есть мы сможем не просто провести анализ, но еще и сравнить его результат с ручной разметкой. Загрузим модель.

In [ ]:
model_path = "./fine_tuned_roberta"

device = torch_directml.device()
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(device)

Определим классификатор

In [ ]:
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Распарсим корпус текстов из файла

In [ ]:
df = pd.read_csv("apple-twitter-sentiment-texts.csv")
data_pairs = list(zip(df["text"], df["sentiment"]))
labels_map = {"LABEL_0": -1, "LABEL_1": 1}

Определим переменные для сохранения данных о точности модели

In [ ]:
true_guesses, false_guesses = 0, 0
confidences = list()

Проведем тестирование. Исключим нейтральные тексты для получения более чистого результата тестирования.

In [ ]:
for text, label in data_pairs:
    if label == 0:
        continue
    result = classifier(text)[0]
    label_id, confidence = result["label"], result["score"]
    confidences.append(confidence)
    status = labels_map.get(label_id, label_id)

    if status == label:
        true_guesses += 1
    else:
        false_guesses += 1

Результаты тестирования на реальных данных:

In [ ]:
print(f"Analysis accuracy: {(true_guesses / (true_guesses + false_guesses)):.2%}")
print(f"Average prediction accuracy: {(sum(confidences) / len(confidences)):.2%}")